In [1]:
import pandas as pd

In [ ]:
en_df = pd.read_csv('data/experiments/English_short_formatted.csv',)
es_df = pd.read_csv('data/experiments/Spanish_short_formatted.csv')

In [8]:
en_df.head()


,Unnamed: 0,Scenario_id,Annotator_id,Scenario,Possible_actions,Selected_action,Moral_values,Cultural_values,Annotator_self_description
0,0,short_id_0_MJI_Emotions,5f6914e252f8200ebfec8422,A doctor is working in an understaffed hospita...,"[""Prioritize the uninsured patient with the se...",1,"{'Care': 3.17, 'Equality': 1.83, 'Proportional...","{'Power Distance': 75, 'Individualism': 35, 'M...",I am a deeply spiritual person who enjoys soli...
1,1,short_id_0_MJI_Emotions,6794102308bc7f868c2acf28,A doctor is working in an understaffed hospita...,"[""Prioritize the uninsured patient with the se...",1,"{'Care': 3.83, 'Equality': 2.17, 'Proportional...","{'Power Distance': 60, 'Individualism': 70, 'M...",It's important to do the right thing.
2,2,short_id_0_MJI_Emotions,59f77c1705d1d60001f14563,A doctor is working in an understaffed hospita...,"[""Prioritize the uninsured patient with the se...",1,"{'Care': 4.33, 'Equality': 2.83, 'Proportional...","{'Power Distance': 25, 'Individualism': -35, '...","despondent communist, doing online surveys due..."
3,3,Reddit_id_166,59d270d6faf42e00012b6cbc,A person is planning their wedding and faces a...,"[""Invite the stepmother's children to the wedd...",2,"{'Care': 4.17, 'Equality': 3.83, 'Proportional...","{'Power Distance': 0, 'Individualism': 0, 'Mas...","I am a introverted person, who likes the simpl..."
4,4,Reddit_id_166,676d69e9d5e21aad658b95fc,A person is planning their wedding and faces a...,"[""Invite the stepmother's children to the wedd...",1,"{'Care': 3.17, 'Equality': 1.83, 'Proportional...","{'Power Distance': -70, 'Individualism': 0, 'M...","I am a software developer and in my free time,..."


In [7]:
en_scenario_ids = set(en_df['Scenario_id'])
es_scenario_ids = set(es_df['Scenario_id'])

# Keep only the ids that are in both dfs
common_scenario_ids = en_scenario_ids & es_scenario_ids
print(f'There are {len(common_scenario_ids)} common scenario ids')

There are 294 common scenario ids


Create an instance of the Example class for each scenario (first row only) and save it to a yaml file.

The criteria used is:
- slug: Scenario_id
- section: (name of the CSV file)
- user: Scenario + Possible_actions
- system: "Here is a scenario description and a list of possible actions. Pick one one of the possible actions. Respond with the number of action (indexed in 1)" 

In [ ]:
import ast
from dataclasses import asdict
import os
import pandas as pd
import yaml
from jlens.examples import Example


def convert_scenarios(csv_file_path: str, output_file: str, description: str) -> list[Example]:
    """Convert scenarios from CSV to a YAML file storing Example specifications."""
    df = pd.read_csv(csv_file_path)
    section_name = os.path.basename(csv_file_path)

    # Keep the first row for each unique scenario ID
    df = df.drop_duplicates(subset=["Scenario_id"], keep="first").sort_values(
        by="Scenario_id"
    )

    examples: list[Example] = []
    system_prompt = (
        "The user will feed you a scenario description and a list of possible actions. "
        "Pick one one of the possible actions. Respond with the number of action only (indexed in 1)"
    )

    for _, row in df.iterrows():
        scenario_id = str(row["Scenario_id"])

        # Safely unwrap stringified lists from the CSV if present
        scenario_raw = row["Scenario"]
        try:
            scenario_parsed = ast.literal_eval(scenario_raw)
            scenario_descr = (
                scenario_parsed[0]
                if isinstance(scenario_parsed, list)
                else str(scenario_parsed)
            )
        except Exception:
            scenario_descr = str(scenario_raw).strip("[]'\"")

        possible_actions = str(row["Possible_actions"])
        user_prompt = f"{scenario_descr}.\n\n{possible_actions}"

        example = Example(
            slug=scenario_id,
            section=section_name,
            description=description,
            user=user_prompt,
            system=system_prompt,
        )
        examples.append(example)

    with open(output_file, "w", encoding="utf-8") as f:
        yaml.safe_dump(
            [asdict(e) for e in examples],
            f,
            sort_keys=False,
            allow_unicode=True,
        )

    print(f"Processed {len(examples)} unique scenarios into {output_file}")
    return examples


def load_examples_from_yaml(yaml_path: str) -> list[Example]:
    """Load the YAML file back into a list of Example instances."""
    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    return [Example(**item) for item in data]

_ = convert_scenarios("data/experiments/English_short_formatted.csv", "data/experiments/English_short_formatted.yaml", "shivaniku/UniMoral")


In [30]:
load_examples_from_yaml("data/experiments/Spanish_short_formatted.yaml")

[Example(slug='Reddit_id_100', section='Spanish_short_formatted.csv', description='shivaniku/UniMoral', prompt=None, system='The user will feed you a scenario description and a list of possible actions. Pick one one of the possible actions. Respond with the number of action only (indexed in 1)', user="Una persona debe elegir entre asistir a un evento previamente planeado con su pareja y celebrar el cumpleaños de su abuela, que fue programado por su familia sin su participación, lo que lleva a un conflicto entre la lealtad a su pareja y las obligaciones familiares.\n\n['Dejar el evento temprano para asistir a la celebración del cumpleaños de la abuela, potencialmente molestando a su pareja', 'Priorizan el evento planeado con su pareja y se pierden la celebración del cumpleaños de la abuela, potencialmente decepcionando a su familia']", assistant_prefill='', n_tracked=None),
 Example(slug='Reddit_id_101', section='Spanish_short_formatted.csv', description='shivaniku/UniMoral', prompt=Non